# StyloScan — tutoriel reproductible

Détecter, quantifier et caractériser l'usage d'IA dans l'écrit scientifique
(articles, thèses, rapports) en **FR / EN / NL**.

Ce notebook décalque la méthodologie de **Bolt, Bui, Chaudhuri & Dexter,
*Stylometry for Latin Literary Criticism*** (Routledge, 2026), transposée de
l'attribution de genre latin à la tâche *humain vs IA* :

| Chapitre (latin) | StyloScan (IA) |
|---|---|
| 26 features interprétables (mots-outils, syntaxe, longueurs) | features interprétables FR/EN/NL (burstiness, connecteurs, tics LLM, variété n-gramme…) |
| vecteur n-dimensionnel par texte | idem |
| Random Forest + validation croisée 5 plis | idem |
| accuracy + F1 | idem |
| importance de Gini (interprétabilité) | idem |
| PCA + violin plots (case studies 3-4) | idem |

> ⚠️ **Aide à l'analyse, pas une preuve.** Faux positifs fréquents sur la prose
> académique formelle, les textes courts et les locuteurs non natifs.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from styloscan import analyze, extract_features
from styloscan.features import FEATURE_CATEGORIES

## 1. Extraction de features et score heuristique (sans entraînement)

In [ ]:
ai_text = ("In today's evolving landscape, it is important to delve into the intricate "
           "realm of writing. Moreover, researchers must navigate a complex tapestry. "
           "Furthermore, robust methods underscore credibility. Additionally, scholars "
           "leverage holistic frameworks. In conclusion, the seamless integration is pivotal.")
human_text = ("I didn't expect that. We ran it again, twice. The first try flopped — bad "
              "reagent. But then it worked, oddly well. Greene hinted at this in 1998. "
              "Nobody followed up. Why? Money ran out. Anyway, the effect is small but real.")

for label, t in [('IA', ai_text), ('humain', human_text)]:
    fr, sc = analyze(t)
    print(f"[{label}] langue={fr.language} indice={sc.score}/100 ({sc.band}) outil~{sc.tool_guess}")

In [ ]:
# Détail des features par catégorie pour le texte IA
fr = extract_features(ai_text, language='en')
for cat, names in FEATURE_CATEGORIES.items():
    print(cat)
    for n in names:
        print(f'   {n:24s} {fr.features[n]:.4f}')

## 2. Voie supervisée : Random Forest + validation croisée + Gini

On fabrique ici un **mini-corpus jouet** pour que le notebook tourne tout seul.
Remplacez-le par votre vrai corpus (`corpus/human/*.txt`, `corpus/ai/*.txt`).

In [ ]:
import os, random, tempfile
random.seed(0)
base = os.path.join(tempfile.gettempdir(), 'styloscan_corpus')
ai_s = ['Moreover, it is important to delve into the intricate realm of the subject.',
        'Furthermore, researchers must navigate a complex tapestry of factors.',
        'Additionally, robust methodologies underscore the credibility of findings.',
        'Consequently, scholars leverage holistic frameworks to foster understanding.',
        'In conclusion, the seamless integration of these pivotal elements is crucial.',
        'It is worth noting that this transformative approach showcases clear synergy.',
        'Overall, the vibrant landscape offers a myriad of opportunities.']
hu_s = ["I didn't expect that.", 'We ran it again, twice.', 'The first try flopped — bad reagent.',
        'But then it worked, oddly well.', 'Greene hinted at this back in 1998.',
        'Nobody followed up though.', 'Why? Money ran out.', 'Anyway, the effect is small but real.',
        'It only shows below twelve degrees.', 'Still, we replicated it.']
for c in ['human', 'ai']:
    os.makedirs(os.path.join(base, c), exist_ok=True)
for i in range(8):
    open(os.path.join(base, 'ai', f'{i}.txt'), 'w').write(' '.join(random.sample(ai_s, 6)))
    open(os.path.join(base, 'human', f'{i}.txt'), 'w').write(' '.join(random.sample(hu_s, 9)))
print('corpus:', base)

In [ ]:
from styloscan.model import train
res = train(base, language='en', n_estimators=300)
print(res.summary())

In [ ]:
from styloscan.viz import plot_gini_importance
from IPython.display import Image
plot_gini_importance(res.gini_importance, 'gini.png')
Image('gini.png')

## 3. PCA des profils stylométriques (case study 3-4 du chapitre)

In [ ]:
from styloscan.model import load_corpus, pca_projection
from styloscan.viz import plot_pca
X, y, classes, paths = load_corpus(base, language='en')
coords, explained, loadings = pca_projection(X)
plot_pca(coords, y, explained, 'pca.png')
Image('pca.png')

## 4. Violin plot d'une feature discriminante

In [ ]:
from styloscan.features import FEATURE_NAMES
from styloscan.viz import plot_violin
feat = 'transition_density'
idx = FEATURE_NAMES.index(feat)
vals = {c: [X[i, idx] for i in range(len(y)) if y[i] == c] for c in classes}
plot_violin(vals, feat, 'violin.png')
Image('violin.png')

## 5. Notes méthodologiques et limites

- **Perplexité / burstiness.** La vraie perplexité exige un modèle de langue. On
  utilise ici des *proxys* sans réseau (burstiness des longueurs de phrase,
  variété n-gramme, ratio de compression). Brancher une perplexité réelle (LM
  local) améliorerait la détection.
- **Calibrage.** Les seuils du score heuristique (`scoring.py`) sont des points
  de départ tirés de la littérature 2023-2025. Dès qu'un corpus étiqueté existe,
  préférez la voie supervisée (Random Forest), qui apprend les seuils sur vos
  données et fournit l'importance de Gini.
- **Faux positifs.** Prose académique très formelle, traductions, textes courts
  et locuteurs non natifs gonflent l'indice. Un score n'est jamais une preuve.